# Projeto: Funções INPE

### Objetivo  
Este notebook implementa funções auxiliares para coletar, extrair e estruturar dados meteorológicos de forma padronizada do INPE. Ele permite gerar DataFrames de previsão para múltiplas cidades brasileiras, com foco em integração com pipelines de dados.

### Informações Técnicas

- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks
 
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 14/05/2025 | Evelise Dias e Lucas Berta     |  1.0   | Criação de funções auxiliares para coleta de dados|
| 16/05/2025 | Gabriel Rost   |  1.1   | Criação de notebooks |
| 19/05/2025 | Evelise Dias   |  1.2   | Inclusão de documentação|


# Configurações Iniciais

In [0]:
%run ../utils_run 


### Helpers (Funções Auxiliares)
Agrupamos funções pequenas e específicas que isolam tarefas comuns e repetitivas, como manipulação de datas, construção de URLs e extração padronizada dos dados da API. Isso torna o código mais legível, evita duplicação e facilita ajustes futuros, sem misturar lógica de negócio com detalhes técnicos da API.

In [0]:
def montar_url_tempo(cidade: str, uf: str, tipo_previsao: str):
    """
    Monta a URL para consulta da API de previsão do tempo do CPTEC-INPE.

    Parâmetros:
    -----------
    cidade : str
        Nome da cidade para consulta.
    uf : str
        Sigla da unidade federativa (estado) da cidade.
    tipo_previsao : str
        Tipo de previsão desejada, por exemplo "hoje" ou "proximos_dias".

    Retorna:
    --------
    str
        URL formatada para consulta da API do CPTEC-INPE.
    """
    cidade_query = quote(f"{cidade}, {uf}")
    return f"https://www.cptec.inpe.br/api/{tipo_previsao}?city={cidade_query}"

In [0]:
def extrair_dados(dados, tipo: str, data_base: str = None):
    
    """
    Extrai e estrutura os dados da resposta da API de previsão do CPTEC-INPE.

    Args:
    -----------
    dados : dict ou list
        Dados brutos recebidos da API para previsão do tempo.
    tipo : str
        Tipo de previsão dos dados recebidos. Valores esperados: "hoje" ou "proximos_dias".
    data_base : str, opcional
        Data base para cálculo das datas das previsões no formato 'YYYY-MM-DD'. Se não fornecida, usa as datas presentes nos dados.

    Return:
    --------
    list of dict
        registros[]
    """
    registros = []
    
    # Organiza os dados da previsão de hoje
    if tipo == "hoje":
        registro = {

            "data_previsao": addDays(data_base, -1) if data_base else dados.get("hora_atual"),
            "temperatura_maxima": f"{dados.get('temperatura_max')}",
            "temperatura_minima": f"{dados.get('temperatura_min')}",
            "precipitacao": f"{dados.get('probabilidade_chuva', 0)}%",
            "fonte": "INPE"
        }
        registros.append(registro)

    elif tipo == "proximos_dias":
        # Organiza os dados dos próximos dias
        if isinstance(dados, dict):
            dados = dados.get("previsao", [])

        if isinstance(dados, list):
            for i, dia in enumerate(dados):
                registro = {
                "data_previsao": addDays(data_base, i) if data_base else dia.get("data_atual"),
                "temperatura_maxima": f"{dia.get('temperatura_max')}",
                "temperatura_minima": f"{dia.get('temperatura_min')}",
                "precipitacao": f"{dia.get('probabilidade_chuva', 0)}%",
                "fonte": "INPE"
                }
                registros.append(registro)

    return registros


### Função principal que coleta previsões
Esta função orquestra a coleta das previsões para todas as capitais, usando os helpers para montar URLs e extrair dados, garantindo que cada etapa fique clara e modular. Ela centraliza a lógica do fluxo de dados, controle de erros e agrega informações locais para que o resultado seja uma lista pronta para análise.

In [0]:
def obter_previsoes_todas_capitais_em_df(nome_enum):
    """
    Coleta previsões do tempo para todas as capitais do enum informado.

    Args:
        nome_enum (str): Nome do enum que contém as capitais e seus estados (UF).

    Returns:
        list: Lista de dicionários com dados de previsão do tempo para cada capital, incluindo
            data, temperaturas, precipitação, fonte, estado e cidade.
    """

    registros = []
    enum_class = getattr(enum_configs, nome_enum)
    data_base = datetime.now().strftime("%Y-%m-%d")

    for uf, dados in enum_class.value.items():
        cidade = dados["capital"]

        try:
            # Monta as URLs da previsão atual e dos próximos dias
            url_hoje = montar_url_tempo(cidade, uf, "forecast-input")
            url_proximos = montar_url_tempo(cidade, uf, "forecast-input-next-days")
            
            resp_hoje = requests.get(url_hoje, timeout=10)
            resp_proximos = requests.get(url_proximos, timeout=10)

            # Processa a resposta da previsão de hoje
            if resp_hoje.status_code == 200:
                dados_hoje = resp_hoje.json()
                registros_hoje = extrair_dados(dados_hoje, "hoje", data_base)
                for resposta in registros_hoje:
                    resposta["estado"] = uf
                    resposta["cidade"] = cidade
                registros.extend(registros_hoje)

            # Processa a resposta da previsão dos próximos dias
            if resp_proximos.status_code == 200:
                dados_proximos = resp_proximos.json()
                registros_proximos = extrair_dados(dados_proximos, "proximos_dias", data_base)
                for resposta in registros_proximos:
                    resposta["estado"] = uf
                    resposta["cidade"] = cidade
                registros.extend(registros_proximos)

        except Exception as e:
            print(f"[ERRO] Falha ao coletar dados para {uf} - {cidade}: {e}")
    
    return registros